# Week 62, Day 4: Shallow and deep copies

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w062d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import copy

grid = [[1, 2], [3, 4]]
shallow = list(grid)
deep = copy.deepcopy(grid)
grid[0].append(99)
grid.append([5])
print(shallow)
print(deep)

row = [0]
pair = [row, row]
twin = copy.deepcopy(pair)
print(twin[0] is twin[1], twin[0] is row)

In [ ]:
import copy

menu = {"tea": ["milk"], "coffee": []}
backup = dict(menu)
backup["juice"] = []                     # the copy's own key
menu["tea"].append("sugar")              # a shared list
print(sorted(backup), backup["tea"], "juice" in menu)

t = (1, 2)
print(tuple(t) is t, copy.copy(t) is t, copy.copy("text") == "text")

class Basket:
    def __init__(self):
        self.items = ["bread"]

b1 = Basket()
b2 = copy.copy(b1)
b2.items.append("jam")
print(b1 is b2, b1.items)

In [ ]:
def naive_deep_copy(obj):
    if isinstance(obj, list):
        return [naive_deep_copy(x) for x in obj]
    if isinstance(obj, dict):
        return {k: naive_deep_copy(v) for k, v in obj.items()}
    if isinstance(obj, set):
        return set(obj)                    # set items are hashable, so usually immutable
    if isinstance(obj, tuple):
        return tuple(naive_deep_copy(x) for x in obj)
    return obj

settings = {"theme": {"colour": "light"}, "recent": ["a.txt"]}
mine = naive_deep_copy(settings)
mine["theme"]["colour"] = "dark"
print(settings["theme"]["colour"], mine["theme"]["colour"])

row = [0]
twin = naive_deep_copy([row, row])
print(twin[0] is twin[1])                  # sharing lost: two separate rows

In [ ]:
def deep_copy(obj, memo=None):
    if memo is None:
        memo = {}
    if id(obj) in memo:
        return memo[id(obj)]
    if isinstance(obj, list):
        new = []
        memo[id(obj)] = new                # register before copying the contents
        new.extend(deep_copy(x, memo) for x in obj)
        return new
    if isinstance(obj, dict):
        new = {}
        memo[id(obj)] = new
        for k, v in obj.items():
            new[deep_copy(k, memo)] = deep_copy(v, memo)
        return new
    return obj

row = [0]
twin = deep_copy([row, row])
print(twin[0] is twin[1], twin[0] is row)

loop = [1]
loop.append(loop)
copied = deep_copy(loop)
print(copied[1] is copied, copied is not loop)

In [ ]:
import copy
import itertools

_ids = itertools.count(1)

class Draft:
    def __init__(self, title, lines, owner):
        self.id = next(_ids)
        self.title, self.lines, self.owner = title, lines, owner
        self._cache = {}

    def __copy__(self):
        new = Draft(self.title, list(self.lines), self.owner)   # its own list of lines
        return new

    def __deepcopy__(self, memo):
        new = Draft.__new__(Draft)
        memo[id(self)] = new
        new.id = next(_ids)                                     # a fresh id
        new.title = self.title
        new.lines = copy.deepcopy(self.lines, memo)
        new.owner = self.owner                                  # shared, never copied
        new._cache = {}                                         # reset
        return new

owner = {"name": "Ada"}
d = Draft("Notes", [["intro"]], owner)
d._cache["words"] = 1
s, dd = copy.copy(d), copy.deepcopy(d)
print(s.id, dd.id, s.lines is d.lines, s.lines[0] is d.lines[0])
print(dd.lines[0] is d.lines[0], dd.owner is owner, dd._cache)

> **Key idea.** A shallow copy (items[:], list(), .copy(), dict(), copy.copy) is a new container holding the same items: changes to the container are separate, changes to mutable items show through both; copying a tuple or string gives the same object. copy.deepcopy copies recursively, reusing immutable atoms, and its memo (id of original to copy, filled in before the contents) preserves sharing and handles cycles. Classes control copying with __copy__ and __deepcopy__(memo), to share, skip or reset attributes. Copying costs time, can't handle files, locks or generators, and is often better replaced by immutable data, copying at boundaries or copy-on-write.

### Exercise: What does the copy share?

Write `shared_parts(original, copied, path="obj")`: walk the two structures side by side (lists, tuples and dict values, matching by index or key) and return a list of paths, depth-first, where the copy holds **the very same mutable object** (list, dict or set) as the original. Don't descend into a shared object (everything inside it is shared too). If the two structures differ in shape at some point, stop there.

In [ ]:
def shared_parts(original, copied, path="obj"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shared():\n    \"shallow versus deep\"\n    import copy\n    data = [[1], {\"a\": [2]}, (3, [4]), 5]\n    assert shared_parts(data, list(data)) == [\"obj[0]\", \"obj[1]\", \"obj[2][1]\"]\n    assert shared_parts(data, data) == [\"obj\"]\n    assert shared_parts(data, copy.deepcopy(data)) == []\n    half = [list(data[0]), data[1], data[2], 5]\n    assert shared_parts(data, half) == [\"obj[1]\", \"obj[2][1]\"]\n    d = {\"x\": {\"y\": [1]}}\n    assert shared_parts(d, {\"x\": dict(d[\"x\"])}) == [\"obj['x']['y']\"]\n    assert shared_parts([1, 2], [1]) == []")

### Exercise: Copies by hand

Without the `copy` module, write `shallow_copy(obj)`: a new list, dict or set with the same items (any other object is returned unchanged), and `deep_copy(obj)`: copies lists, dicts and sets recursively and rebuilds tuples, returning **the same tuple** when nothing inside it needed copying. Don't worry about sharing or cycles yet.

In [ ]:
def shallow_copy(obj):
    ___

def deep_copy(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_handmade():\n    \"shallow and deep by hand\"\n    data = {\"a\": [1, [2]], \"b\": (3, [4]), \"c\": (5, 6), \"d\": {7}}\n    s = shallow_copy(data)\n    assert s == data and s is not data and s[\"a\"] is data[\"a\"]\n    d = deep_copy(data)\n    assert d == data and d[\"a\"] is not data[\"a\"] and d[\"a\"][1] is not data[\"a\"][1]\n    assert d[\"b\"] is not data[\"b\"] and d[\"b\"][1] is not data[\"b\"][1]\n    assert d[\"c\"] is data[\"c\"] and d[\"d\"] is not data[\"d\"] and d[\"d\"] == {7}\n    assert shallow_copy(\"text\") == \"text\" and deep_copy(5) == 5\n    assert not uses(source, \"copy.deepcopy\") and not uses(source, \"deepcopy\")")

### Exercise: Keeping the shape

Write `clone(obj, memo=None)`, a deep copy of lists and dicts (keys reused, values copied) that **preserves sharing** (an object that appears twice in the original appears as one copy, twice) and **copes with cycles**, using a memo dict from `id` to copy. Other objects are returned unchanged. Then `same_shape(a, b)`: True if two list structures have the same sharing pattern: walk them side by side and check that whenever one has seen an inner list before, the other has seen the corresponding list at the same moment (use a pair of id-to-position dicts).

In [ ]:
def clone(obj, memo=None):
    ___

def same_shape(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memo():\n    \"the memo\"\n    row = [0]\n    grid = [row, row, [0]]\n    c = clone(grid)\n    assert c == grid and c[0] is c[1] and c[0] is not row and c[2] is not c[0]\n    loop = {\"name\": \"me\"}\n    loop[\"self\"] = loop\n    lc = clone(loop)\n    assert lc[\"self\"] is lc and lc is not loop\n    a = [1]\n    a.append(a)\n    b = clone(a)\n    assert b[1] is b and b[0] == 1\n    assert same_shape(grid, c) and not same_shape(grid, [[0], [0], [0]])\n    assert same_shape(a, b) and not same_shape([row, row], [[0], [0]])\n    assert same_shape([[1], [2]], [[1], [2]])")

### Exercise: Choosing what gets copied

Write a class `Account(owner, history)`: `owner` is shared with every copy (it's a person, not data), `history` is a list of transaction lists like `["in", 50]`, and `cache` starts as an empty dict. Define `__copy__` so a shallow copy gets its **own** outer history list (sharing the transaction lists) and the same cache dict, and `__deepcopy__(memo)` so a deep copy deep-copies the history (passing the memo on), shares the owner and gets a **fresh empty** cache. Each account gets a number from a class-level counter starting at 1, and copies get new numbers too.

In [ ]:
import copy
import itertools

class Account:
    _numbers = itertools.count(1)

    def __init__(self, owner, history):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dunder():\n    \"__copy__ and __deepcopy__\"\n    import copy\n    ada = {\"name\": \"Ada\"}\n    acc = Account(ada, [[\"in\", 50], [\"out\", 20]])\n    acc.cache[\"balance\"] = 30\n    s = copy.copy(acc)\n    assert s.owner is ada and s.history is not acc.history and s.history[0] is acc.history[0]\n    assert s.cache is acc.cache and s.number != acc.number\n    d = copy.deepcopy(acc)\n    assert d.owner is ada and d.history == acc.history and d.history[0] is not acc.history[0]\n    assert d.cache == {} and d.number not in (acc.number, s.number)\n    pair = copy.deepcopy([acc, acc])\n    assert pair[0] is pair[1] and pair[0] is not acc\n    assert Account(ada, []).number > d.number")

### Exercise: Snapshots for undo

Write `History(state)` for undo. `current()` returns a **deep copy** of the current state (so callers can't change history by mutating it); `change(func)` deep-copies the current state, calls `func(copy)` (which mutates it), and stores the result as the new current state; `undo()` goes back one step (raising `IndexError` if there's nothing to undo); `steps()` is how many changes can be undone. The starting state is copied too.

In [ ]:
import copy

class History:
    def __init__(self, state):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_undo():\n    \"deep snapshots\"\n    start = {\"items\": [\"tea\"], \"total\": 1}\n    h = History(start)\n    start[\"items\"].append(\"hack\")\n    assert h.current() == {\"items\": [\"tea\"], \"total\": 1}\n    def add_jam(s):\n        s[\"items\"].append(\"jam\")\n        s[\"total\"] += 1\n    h.change(add_jam)\n    h.change(add_jam)\n    assert h.current()[\"items\"] == [\"tea\", \"jam\", \"jam\"] and h.steps() == 2\n    view = h.current()\n    view[\"items\"].clear()\n    assert h.current()[\"total\"] == 3\n    h.undo()\n    assert h.current() == {\"items\": [\"tea\", \"jam\"], \"total\": 2}\n    h.undo()\n    assert h.current()[\"items\"] == [\"tea\"] and h.steps() == 0\n    try:\n        h.undo()\n        assert False\n    except IndexError:\n        pass")

### Bug hunt: A copy that changed the original

Each user starts from a copy of the default settings, but `dict(...)` copies only the outer dict, so changing the user's theme also changes the defaults: this prints `dark dark`. Make the copy deep so it prints `light dark`.

In [ ]:
import copy

DEFAULTS = {"theme": {"mode": "light"}, "font": 12}
user = dict(DEFAULTS)
user["theme"]["mode"] = "dark"
print(DEFAULTS["theme"]["mode"], user["theme"]["mode"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"deep copy\"\n    expect_output(\"light dark\")")

### Bug hunt: One list for every key

`dict.fromkeys(names, [])` makes **one** empty list and uses it as the value for every key, so adding to Ada's list adds to Ben's too: this prints `{'ada': ['tea'], 'ben': ['tea']}`. Give each key its own list so it prints `{'ada': ['tea'], 'ben': []}`.

In [ ]:
names = ["ada", "ben"]
orders = dict.fromkeys(names, [])
orders["ada"].append("tea")
print(orders)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate lists\"\n    expect_output(\"{'ada': ['tea'], 'ben': []}\")")

### Bug hunt: A shallow copy of an object

`copy.copy` of a `Tally` makes a new object but shares its `marks` list, so marking the copy marks the original: this prints `[1, 2] [1, 2]`. Give `Tally` a `__copy__` method that copies the list so it prints `[1] [1, 2]`.

In [ ]:
import copy

class Tally:
    def __init__(self):
        self.marks = []

original = Tally()
original.marks.append(1)
spare = copy.copy(original)
spare.marks.append(2)
print(original.marks, spare.marks)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"__copy__\"\n    expect_output(\"[1] [1, 2]\")")

### Puzzle: A __deepcopy__ that keeps the owner

Arrange the lines of a `__deepcopy__` method that copies the data but shares the owner. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # a = [[1]]; b = a[:]; b[0].append(2); a
q = ___   # a = [[1]]; b = a[:]; b.append([3]); len(a)
r = ___   # True or False: t = (1, 2); copy.copy(t) is t
s = ___   # r = [0]; c = copy.deepcopy([r, r]); c[0] is c[1]
t = ___   # the exception from copy.deepcopy(x for x in [1]), as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [[1, 2]] and q == 1 and r is True, \"p, q and r\"\n    assert s is True and t == \"TypeError\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: your own deepcopy

Write `deep_copy(obj, memo=None)` that does most of what `copy.deepcopy` does:

- atoms (`int`, `float`, `complex`, `bool`, `str`, `bytes`, `None`), functions and classes are returned as they are;
- lists, dicts (keys and values) and sets are copied, preserving sharing and cycles with a memo;
- tuples are rebuilt only if some item changed (otherwise the same tuple is returned), and recorded in the memo;
- if the object's class defines `__deepcopy__`, call it with the memo and use the result;
- other objects are copied by making a new instance with `cls.__new__(cls)`, registering it in the memo, and deep-copying each attribute in `vars(obj)` onto it.

Don't import `copy`.

In [ ]:
def deep_copy(obj, memo=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_containers():\n    \"containers, sharing and cycles\"\n    row = [1]\n    data = {\"rows\": [row, row], \"tags\": {\"a\"}, \"pair\": (1, \"x\"), \"mixed\": (2, [3])}\n    c = deep_copy(data)\n    assert c == data and c[\"rows\"][0] is c[\"rows\"][1] and c[\"rows\"][0] is not row\n    assert c[\"pair\"] is data[\"pair\"] and c[\"mixed\"] is not data[\"mixed\"] and c[\"mixed\"][1] is not data[\"mixed\"][1]\n    assert c[\"tags\"] == {\"a\"} and c[\"tags\"] is not data[\"tags\"]\n    loop = []\n    loop.append(loop)\n    assert deep_copy(loop)[0] is not loop\n    lc = deep_copy(loop)\n    assert lc[0] is lc\n\ndef test_objects():\n    \"objects and __deepcopy__\"\n    class Node:\n        def __init__(self, value):\n            self.value, self.next = value, None\n    a, b = Node([1]), Node([2])\n    a.next, b.next = b, a\n    ca = deep_copy(a)\n    assert isinstance(ca, Node) and ca is not a and ca.value == [1] and ca.value is not a.value\n    assert ca.next.next is ca and ca.next is not b\n    class Shared:\n        def __deepcopy__(self, memo):\n            return self\n    s = Shared()\n    out = deep_copy([s, Node(s)])\n    assert out[0] is s and out[1].value is s\n    def f():\n        pass\n    assert deep_copy([f, Node])[0] is f and deep_copy([f, Node])[1] is Node\n    assert not uses(source, \"copy\")")

## Warm-up

The shallow copy has its own outer list (so it didn't get `[5]`) but shares the rows (so it did get `99`). The deep copy shares nothing with the original, yet it keeps the **shape** of the original: one row, appearing twice.

## Shallow copies

A **shallow copy** is a new container holding **the same objects** as the original. Python offers many ways to make one:

| type | shallow copy |
|---|---|
| list | `items[:]`, `list(items)`, `items.copy()`, `[*items]` |
| dict | `dict(d)`, `d.copy()`, `{**d}` |
| set | `set(s)`, `s.copy()` |
| anything | `copy.copy(obj)` |

Adding, removing or replacing items in the copy doesn't affect the original. But the items are shared, so **changing** a mutable item (appending to an inner list, setting an attribute on an object) shows through both. For a flat list of numbers or strings, a shallow copy is all you need, because the items are immutable.

Tuples, strings and frozensets are special: since they can't change, "copying" them gives back **the same object** (`tuple(t) is t`, `copy.copy(t) is t`). There's nothing to protect.

`copy.copy` of an instance of your own class makes a new instance whose `__dict__` is a shallow copy of the original's: new object, same attribute values.

## Deep copies

`copy.deepcopy(obj)` copies **recursively**: containers inside containers, objects' attributes, everything mutable, so the result shares nothing mutable with the original. Immutable atoms (numbers, strings, `None`) are reused, since that's safe. A tuple is rebuilt only if something inside it needed copying.

Deep copies are what you want for a **snapshot**: an undo history, a "before" state in a test, a transaction to roll back, a default configuration that each user modifies. Hand-written, the idea is a few lines of recursion, though as you'll see next, this simple version gets two things wrong:

## The memo

The naive version has two problems: it turns **one shared** object into **several copies**, and on a **cycle** (a list that contains itself) it recurses forever. `deepcopy` solves both with a **memo**: a dict from `id(original)` to its copy.

1. Before copying an object, look up its id in the memo. If it's there, return the **existing copy**: sharing is preserved.
2. Otherwise, make the **empty** new container, put it in the memo **at once**, and only then copy the contents into it. If a cycle leads back to this object, step 1 finds the half-built copy and returns it, closing the cycle in the copy.

(`deepcopy` also keeps the originals alive in the memo during the copy, so that ids can't be reused halfway through.)

## Copying your own classes

By default, `copy.copy` and `copy.deepcopy` copy an instance's `__dict__` (shallowly or deeply). When that's wrong, a class can take control:

- `__copy__(self)`: return the shallow copy. Useful when "shallow" should still give the copy its own list, say.
- `__deepcopy__(self, memo)`: return the deep copy. Pass `memo` on to `copy.deepcopy` for the parts you do copy, and put `memo[id(self)] = new` in before copying parts that might refer back.

Common reasons: some attributes must **not** be copied (a lock, an open file, a database connection, a reference to a shared registry or the owning user), some should be **reset** (a cache, a fresh id number), or copying is cheaper done by hand. (Week 31's `__slots__` classes and frozen dataclasses work with `copy` too; Python 3.13 also added `copy.replace` for "a copy with these fields changed".)

## When not to copy

Copying is a blunt tool:

- **It costs time and memory** in proportion to the size of the structure. Deep-copying a large state on every change can dominate a program.
- **Some things can't or mustn't be copied**: files, sockets, locks, generators (`deepcopy` of a generator raises `TypeError`), and objects whose identity matters (a user, a registry).
- **It hides design questions**: if you're copying "just in case", decide who owns the data.

Alternatives:

- **Immutable data** (yesterday): nothing to protect, so share freely; "changes" build new objects that can **share unchanged parts** with the old ones (a new tuple of mostly the same items).
- **Copy on the boundary**: copy once when data comes into an object (`self.items = list(items)`) and when it goes out, not everywhere.
- **Copy-on-write**: share until someone wants to change it, and copy only then (Friday's challenge style, and how many databases and the `fork` system call work).

## Practice

## Quiz

## Daily challenge

## Before you go

Find one place in your own code where you copy "just in case", and decide who really owns that data. Tomorrow: weak references, which let you refer to an object without keeping it alive.